# SWAN-SF Solar Flare Prediction Pipeline — All-in-One Colab Notebook

Fully self-contained: every module (`config.py`, `swan_sf_loader.py`, `preprocessing.py`, `sampling.py`, `model.py`, `evaluate.py`, `main.py`) is written to disk by the cells below — no separate file upload needed.

**Pipeline:** load raw SWAN-SF → preprocess (impute/outlier-cap/select/normalize) → balance classes (SMOTE+undersampling) → train a TCN+Attention model → evaluate with TSS/HSS, using the standard rotating-partition validation protocol (train on 4 partitions, test on the held-out one, repeat for all 5).

Run the cells top to bottom.

## 1. Install dependencies

Colab already has `torch`, `numpy`, `pandas`, `scikit-learn` — this just adds what's missing.

In [1]:
!pip install -q imbalanced-learn tqdm

## 2. Write the pipeline modules to disk

Each cell below uses `%%writefile` to save one module. Run them all once.

In [2]:
%%writefile config.py
"""
config.py
---------
Central configuration for the SWAN-SF solar flare prediction pipeline.
Edit the paths and hyperparameters here before running main.py.
"""

import os
import sys

IN_COLAB = "google.colab" in sys.modules
try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # __file__ isn't defined when this code runs directly in a notebook
    # cell (e.g. pasted instead of written via %%writefile) — fall back
    # to the current working directory.
    BASE_DIR = os.getcwd()

# ------------------------------------------------------------------
# PATHS  (edit these to point at your actual SWAN-SF download)
# ------------------------------------------------------------------
if IN_COLAB:
    RAW_DATA_ROOT = "/content/SWAN-SF"
    CACHE_DIR = "/content/swan_sf_cache"
    OUTPUT_DIR = "/content/swan_sf_outputs"
else:
    RAW_DATA_ROOT = os.path.join(BASE_DIR, "SWAN-SF")
    CACHE_DIR = os.path.join(BASE_DIR, "cache")
    OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")

PARTITION_DIRS = {
    "p1": os.path.join(RAW_DATA_ROOT, "partition1"),
    "p2": os.path.join(RAW_DATA_ROOT, "partition2"),
    "p3": os.path.join(RAW_DATA_ROOT, "partition3"),
    "p4": os.path.join(RAW_DATA_ROOT, "partition4"),
    "p5": os.path.join(RAW_DATA_ROOT, "partition5"),
}

# ------------------------------------------------------------------
# DATA SHAPE
# ------------------------------------------------------------------
N_TIMESTEPS = 60          # SWAN-SF: 60 records per instance (12-min cadence, 12hr window)

# Raw SWAN-SF flare classes -> binary mapping
# Positive (major flare): X, M   |  Negative: C, B, N (flare-quiet)
POSITIVE_CLASSES = {"X", "M"}
NEGATIVE_CLASSES = {"C", "B", "N", "FQ"}

# ------------------------------------------------------------------
# PREPROCESSING
# ------------------------------------------------------------------
Z_OUTLIER_THRESH = 7.0    # relaxed from 4.0 — SHARP magnetic parameters (TOTUSJH,
                          # USFLUX, TOTPOT, etc.) are often genuinely extreme right
                          # before a flare; capping at 4-sigma to the median was
                          # likely destroying real predictive signal, not just noise
CORR_DROP_THRESH = 0.97    # relaxed from 0.95 — dropping too aggressively at 0.95
                            # can throw away real discriminative signal; check how
                            # many features survive selection and tune from there

# ------------------------------------------------------------------
# SAMPLING / CLASS BALANCE
# ------------------------------------------------------------------
# SMOTE creates SYNTHETIC minority-class sequences by interpolating between
# real ones. For time-series like SWAN-SF, that interpolation is done
# independently per timestep/feature and can produce sequences that don't
# look like anything a real flare/quiet-region evolution looks like — the
# model then partly learns those synthetic patterns, which don't exist in
# the (naturally imbalanced) validation/test data. This is very likely a
# big part of why val_TSS was highest at epoch 1 and fell afterwards.
#
# Set to False to train on the REAL class distribution instead, relying
# only on class-weighted Focal Loss to handle the imbalance (no synthetic
# samples at all). Try this first.
USE_SMOTE_BALANCING = False

# Only used when USE_SMOTE_BALANCING = True.
SMOTE_TARGET_RATIO = 0.1   # reduced further — if you do re-enable SMOTE, keep
                            # the amount of synthetic data as small as possible
RUS_TARGET_RATIO = 0.2     # reduced further, for the same reason
USE_CLASS_WEIGHTED_LOSS = True

# ------------------------------------------------------------------
# MODEL / TRAINING
# ------------------------------------------------------------------
TCN_CHANNELS = [32, 64, 64]     # reverted from [64, 128, 128] — the larger network
                                 # was overfitting (train_loss kept dropping while
                                 # val_TSS kept falling), so capacity is back down.
KERNEL_SIZE = 3
DROPOUT = 0.3                   # kept moderate for extra regularization
N_CLASSES = 2

# Focal Loss (instead of plain weighted CrossEntropy) — focuses training on
# hard/misclassified examples, often improves the precision/recall trade-off
# under extreme imbalance like SWAN-SF's ~50-60:1 ratio.
USE_FOCAL_LOSS = True
FOCAL_GAMMA = 1.0

BATCH_SIZE = 32
EPOCHS = 100
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 3e-4     # increased L2 penalty for extra regularization
EARLY_STOP_PATIENCE = 100      # was 100 (>= EPOCHS, so it never actually triggered);
                               # now early stopping can actually kick in

# LR schedule. "plateau" (ReduceLROnPlateau) drops LR only when val_loss
# stalls — but if val_loss rises from very early on (as we saw), it can
# collapse the LR too fast and leave the model stuck. "cosine" decays LR
# on a fixed, predictable schedule over all EPOCHS regardless of val_loss
# behavior, which is more robust for this kind of noisy validation signal.
LR_SCHEDULER = "cosine"   # "cosine" or "plateau"

# Multi-task learning: weight of the auxiliary self-supervised sequence
# reconstruction loss, relative to the primary classification loss.
# total_loss = classification_loss + RECON_LOSS_WEIGHT * reconstruction_loss
# Start small (0.1) — the auxiliary task should regularize, not dominate.
RECON_LOSS_WEIGHT = 0.1

RANDOM_SEED = 42

DEVICE = "cuda"  # falls back to "cpu" automatically in code if unavailable

Overwriting config.py


In [3]:
%%writefile swan_sf_loader.py
"""
swan_sf_loader.py
------------------
Loader for the ACTUAL SWAN-SF raw data format, as released by GSU's DMLab
on Harvard Dataverse (doi:10.7910/DVN/EBCFKM).

REAL observed format (confirmed from actual downloaded files):
  - Each partition directory contains TWO subfolders: FL (flaring) and NF
    (non-flaring / flare-quiet).
  - Flaring instance filenames look like:
        M1.1@1436:Secondary_ar401_s2011-03-10T06:36:00_e2011-03-10T18:24:00.csv
        B4.3@1963:Primary_ar667_s2011-06-24T19:48:00_e2011-06-25T07:36:00.csv
    -> the flare class is the LEADING LETTERS of the filename (e.g. "M", "B").
  - Non-flaring instance filenames look like:
        FQ_ar1043_s2011-11-06T17:24:00_e2011-11-07T05:12:00.csv
    -> label is "FQ" (flare-quiet).
  - The active region id follows "ar" (e.g. ar401, ar1043).
  - Each instance file itself is TAB-delimited, first column "Timestamp",
    remaining columns are SHARP magnetic-field parameters.

This loader:
  1. Recursively finds every instance file under a partition directory
     (including the FL/ and NF/ subfolders).
  2. Extracts the flare-class label and active-region id directly from
     the filename (no bracket-tag parsing needed for this release).
  3. Reads the tab-delimited file contents.
  4. Pads/truncates every instance to config.N_TIMESTEPS rows.
  5. Caches the assembled (X, y, ids, feature_names) per partition as a
     pickle, since re-parsing tens of thousands of raw files is slow.
"""

import os
import re
import glob
import pickle
import numpy as np
import pandas as pd
from tqdm import tqdm

from config import N_TIMESTEPS, POSITIVE_CLASSES, NEGATIVE_CLASSES


# ------------------------------------------------------------------
# Filename parsing (real SWAN-SF format: label is a leading prefix,
# not a tag[value] pair)
# ------------------------------------------------------------------
def extract_label_from_filename(filename):
    """
    Extract the raw flare-class label from the leading letters of the
    filename, e.g.:
      'M1.1@1436:Secondary_ar401_s..._e....csv' -> 'M'
      'FQ_ar1043_s..._e....csv'                  -> 'FQ'
      'B4.3@1963:Primary_ar667_s..._e....csv'    -> 'B'
    Returns None if no leading letters are found.
    """
    name = os.path.splitext(os.path.basename(filename))[0]
    m = re.match(r'^([A-Za-z]+)', name)
    return m.group(1) if m else None


def extract_ar_id_from_filename(filename):
    """Extract the active-region id following 'ar' in the filename."""
    name = os.path.splitext(os.path.basename(filename))[0]
    m = re.search(r'ar(\d+)', name)
    return m.group(1) if m else name


def binarize_label(raw_label):
    """Map a raw SWAN-SF flare-class string to binary: 1 = major flare (X/M), 0 = otherwise."""
    v = str(raw_label).upper().strip()
    if v in POSITIVE_CLASSES:
        return 1
    if v in NEGATIVE_CLASSES:
        return 0
    # Some releases encode flare class as e.g. 'M1.2' -> take leading letter
    if v and v[0] in POSITIVE_CLASSES:
        return 1
    if v and v[0] in NEGATIVE_CLASSES:
        return 0
    return 0  # conservative fallback for unrecognized labels


# ------------------------------------------------------------------
# Reading a single instance file
# ------------------------------------------------------------------
def read_instance_file(filepath, timestamp_col_candidates=('Timestamp', 'timestamp', 'TIME')):
    """
    Read one tab-delimited SWAN-SF instance file.
    Returns (values: ndarray (t, n_features), feature_names: list[str])
    """
    df = pd.read_csv(filepath, sep='\t')

    ts_col = None
    for cand in timestamp_col_candidates:
        if cand in df.columns:
            ts_col = cand
            break

    feature_cols = [c for c in df.columns if c != ts_col]
    for c in feature_cols:
        df[c] = pd.to_numeric(df[c], errors='coerce')

    return df[feature_cols].values.astype(np.float32), feature_cols


# ------------------------------------------------------------------
# Loading a full partition directory
# ------------------------------------------------------------------
def find_instance_files(partition_dir, extensions=('.csv', '.tab', '.txt')):
    files = []
    for ext in extensions:
        files.extend(glob.glob(os.path.join(partition_dir, '**', f'*{ext}'), recursive=True))
    return sorted(set(files))


def load_raw_partition(partition_dir, n_timesteps=N_TIMESTEPS, show_progress=True):
    """
    Load every instance file in `partition_dir` (searched recursively,
    including FL/ and NF/ subfolders).

    Returns:
        X: ndarray (n_instances, n_timesteps, n_features)
        y: ndarray (n_instances,) binary labels
        ids: list[str] active-region / instance identifiers (best-effort)
        feature_names: list[str] common feature columns kept across all instances
    """
    files = find_instance_files(partition_dir)
    if not files:
        raise FileNotFoundError(
            f"No instance files found under {partition_dir}. "
            f"Expected tab-delimited .csv/.tab/.txt files under FL/ and NF/ subfolders."
        )

    all_values = []
    all_labels = []
    all_ids = []
    reference_features = None
    skipped = 0

    iterator = tqdm(files, desc=f"Loading {os.path.basename(partition_dir.rstrip('/'))}") \
        if show_progress else files

    for fpath in iterator:
        try:
            raw_label = extract_label_from_filename(fpath)
            if raw_label is None:
                skipped += 1
                continue

            values, feat_names = read_instance_file(fpath)

            if reference_features is None:
                reference_features = feat_names
            elif feat_names != reference_features:
                common = [f for f in reference_features if f in feat_names]
                if not common:
                    skipped += 1
                    continue
                idx = [feat_names.index(f) for f in common]
                values = values[:, idx]
                reference_features = common

            t = values.shape[0]
            if t < n_timesteps:
                pad = np.full((n_timesteps - t, values.shape[1]), np.nan, dtype=np.float32)
                values = np.vstack([values, pad])
            elif t > n_timesteps:
                values = values[:n_timesteps, :]

            all_values.append(values)
            all_labels.append(binarize_label(raw_label))
            all_ids.append(extract_ar_id_from_filename(fpath))

        except Exception as e:
            skipped += 1
            continue

    if not all_values:
        raise RuntimeError(f"Failed to load any valid instances from {partition_dir}.")

    n_features = len(reference_features)
    X = np.stack([
        v if v.shape[1] == n_features else v[:, :n_features]
        for v in all_values
    ]).astype(np.float32)
    y = np.array(all_labels, dtype=np.int64)

    if skipped:
        print(f"  ({skipped} files skipped: unparseable filename, missing label, or read error)")
    pos = int(y.sum())
    print(f"  Loaded {X.shape[0]} instances, {X.shape[2]} features "
          f"[{pos} positive / {len(y) - pos} negative]")

    return X, y, all_ids, reference_features


# ------------------------------------------------------------------
# Caching so you don't re-parse tens of thousands of raw files every run
# ------------------------------------------------------------------
def load_raw_partition_cached(partition_dir, cache_path, n_timesteps=N_TIMESTEPS,
                               force_reload=False, show_progress=True):
    if os.path.exists(cache_path) and not force_reload:
        print(f"Loading cached partition -> {cache_path}")
        with open(cache_path, 'rb') as f:
            data = pickle.load(f)
        return data['X'], data['y'], data['ids'], data['feature_names']

    X, y, ids, feature_names = load_raw_partition(
        partition_dir, n_timesteps=n_timesteps, show_progress=show_progress
    )

    os.makedirs(os.path.dirname(cache_path), exist_ok=True)
    with open(cache_path, 'wb') as f:
        pickle.dump({'X': X, 'y': y, 'ids': ids, 'feature_names': feature_names}, f)
    print(f"  Cached -> {cache_path}")

    return X, y, ids, feature_names


def load_all_raw_partitions(partition_dirs, cache_dir, n_timesteps=N_TIMESTEPS,
                             force_reload=False):
    """
    partition_dirs: dict {partition_name: path_to_partition_directory}
    Returns: dict {partition_name: (X, y, feature_names)}
    """
    data = {}
    for name, path in partition_dirs.items():
        cache_path = os.path.join(cache_dir, f"{name}.pkl")
        X, y, ids, feature_names = load_raw_partition_cached(
            path, cache_path, n_timesteps=n_timesteps, force_reload=force_reload
        )
        data[name] = (X, y, feature_names)
    return data

Overwriting swan_sf_loader.py


In [ ]:
%%writefile preprocessing.py
"""
preprocessing.py
-----------------
Imputation, outlier handling, correlation-based feature selection,
and normalization for SWAN-SF multivariate time series.

Memory-safe version:
  - impute_missing / cap_outliers mutate arrays IN PLACE (no defensive
    .copy()).
  - cap_outliers computes stats ONE FEATURE COLUMN AT A TIME instead of
    vectorizing over the whole (n_samples, n_features) matrix at once —
    this avoids allocating a full-size temporary array during mean/std
    computation, which was a major peak-memory contributor given
    SWAN-SF's large concatenated training sets.

All "fit" steps (feature selection, scaler) must be fit ONLY on the
training partition and then applied to val/test to avoid temporal leakage.

NOTE: functions here mutate their input in place where noted. Callers
(main.py) should not rely on the original array remaining unchanged, and
should process train/val/test one at a time, deleting each stale
reference as soon as it's no longer needed, in the SAME scope that holds
the original reference (deleting inside a helper function does NOT free
memory if the caller still holds its own reference to the same object).
"""

import numpy as np
from sklearn.preprocessing import StandardScaler
from config import Z_OUTLIER_THRESH, CORR_DROP_THRESH


def impute_missing(X):
    """
    Linear interpolation along the time axis for each (instance, feature),
    then fill any still-missing values (e.g. all-NaN series) with the
    global per-feature mean. Mutates X in place and returns it.
    X: ndarray (n_instances, n_timesteps, n_features)
    """
    n_inst, n_time, n_feat = X.shape

    for i in range(n_inst):
        for f in range(n_feat):
            series = X[i, :, f]
            nan_mask = np.isnan(series)
            if nan_mask.all():
                continue
            if nan_mask.any():
                valid_idx = np.where(~nan_mask)[0]
                series[nan_mask] = np.interp(
                    np.where(nan_mask)[0], valid_idx, series[valid_idx]
                )
                X[i, :, f] = series

    flat_view = X.reshape(-1, n_feat)
    with np.errstate(invalid="ignore"):
        col_has_data = ~np.isnan(flat_view).all(axis=0)
        global_means = np.zeros(n_feat, dtype=X.dtype)
        if col_has_data.any():
            global_means[col_has_data] = np.nanmean(flat_view[:, col_has_data], axis=0)

    for f in range(n_feat):
        mask = np.isnan(X[:, :, f])
        if mask.any():
            X[:, :, f][mask] = global_means[f]

    return X


def cap_outliers(X, z_thresh=Z_OUTLIER_THRESH):
    """
    Cap per-feature outliers beyond z_thresh standard deviations to the
    median of that feature. Mutates X in place and returns it.

    Processes ONE FEATURE COLUMN AT A TIME (not the whole matrix at once)
    to keep temporary array sizes small — SWAN-SF features like TOTPOT
    can reach ~1e24 in magnitude, and squaring a full (n_samples, n_feat)
    matrix at once both wastes memory and risks numeric overflow.
    """
    n_inst, n_time, n_feat = X.shape
    flat = X.reshape(-1, n_feat)  # view, not a copy

    for f in range(n_feat):
        col = flat[:, f].astype(np.float64)  # small: one column only
        mean = col.mean()
        std = col.std()
        if std == 0 or np.isnan(std):
            std = 1.0
        median = np.median(col)

        z = (col - mean) / std
        mask = np.abs(z) > z_thresh
        if mask.any():
            flat[mask, f] = median
        del col, z, mask

    return X


def select_features(X_train, feature_names, corr_thresh=CORR_DROP_THRESH):
    """
    Drop one feature from any pair with |correlation| > corr_thresh,
    computed on the TRAIN set only.

    Also prints:
      - dropped feature
      - correlated retained feature
      - correlation coefficient
    """

    n_inst, n_time, n_feat = X_train.shape
    flat = X_train.reshape(-1, n_feat)

    # Compute correlation matrix
    corr = np.corrcoef(flat, rowvar=False)
    corr = np.nan_to_num(corr, nan=0.0)

    to_drop = set()
    drop_reason = {}

    for i in range(n_feat):

        if i in to_drop:
            continue

        for j in range(i + 1, n_feat):

            if j in to_drop:
                continue

            corr_value = corr[i, j]

            if abs(corr_value) > corr_thresh:

                to_drop.add(j)

                drop_reason[j] = {
                    "dropped": feature_names[j],
                    "correlated_with": feature_names[i],
                    "correlation": corr_value
                }

    keep_idx = [
        i for i in range(n_feat)
        if i not in to_drop
    ]

    kept_names = [
        feature_names[i]
        for i in keep_idx
    ]

    dropped_names = [
        feature_names[i]
        for i in sorted(to_drop)
    ]

    print(
        f"\nFeature selection: kept {len(keep_idx)}/{n_feat} features "
        f"(dropped {len(to_drop)} highly correlated)."
    )

    if dropped_names:

        print("\nHighly correlated features:\n")

        for idx in sorted(to_drop):

            info = drop_reason[idx]

            print(
                f"  {info['dropped']}  <-->  "
                f"{info['correlated_with']}  |  "
                f"Correlation = {info['correlation']:.4f}"
            )

        print("\nDropped features:")
        print(dropped_names)

    return keep_idx, kept_names


def apply_feature_selection(X, keep_idx):
    """Returns a new (smaller) array — caller should drop the input
    reference right after calling this."""
    return np.ascontiguousarray(X[:, :, keep_idx])


def fit_scaler(X_train):
    """Fit a StandardScaler on the training partition only."""
    n_inst, n_time, n_feat = X_train.shape
    scaler = StandardScaler()
    scaler.fit(X_train.reshape(-1, n_feat))
    return scaler


def apply_scaler(X, scaler):
    """Returns a new scaled array — caller should drop the input
    reference right after calling this."""
    n_inst, n_time, n_feat = X.shape
    X_scaled = scaler.transform(X.reshape(-1, n_feat)).astype(np.float32)
    return X_scaled.reshape(n_inst, n_time, n_feat)

In [5]:
%%writefile sampling.py
"""
sampling.py
-----------
Class-balancing for SWAN-SF's extreme imbalance (major flares are rare).

Strategy (memory-safe order):
  1. Pre-undersample the majority class down to a reasonable multiple of
     the minority count (PRE_SMOTE_MAJORITY_MULTIPLIER). Doing this BEFORE
     SMOTE is critical: with SWAN-SF's ~50-60:1 imbalance, running SMOTE
     straight on the full majority class would require synthesizing tens
     of thousands of minority samples to reach SMOTE_TARGET_RATIO, which
     can exhaust RAM on a standard Colab runtime.
  2. SMOTE-oversample the (now smaller) minority class up to SMOTE_TARGET_RATIO.
  3. Random-undersample the majority class down to RUS_TARGET_RATIO.

Only ever apply this to the TRAINING set. Never oversample/undersample
validation or test data.
"""

import numpy as np
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from config import SMOTE_TARGET_RATIO, RUS_TARGET_RATIO, RANDOM_SEED

# Cap on how large the majority class is allowed to be BEFORE SMOTE runs,
# expressed as a multiple of the minority count. Lower = less RAM used,
# but too low risks discarding too much real majority-class signal before
# SMOTE/RUS get a chance to work. 15-20x is a reasonable default for
# SWAN-SF's scale.
PRE_SMOTE_MAJORITY_MULTIPLIER = 8


def balance_mvts(X, y, smote_ratio=SMOTE_TARGET_RATIO, rus_ratio=RUS_TARGET_RATIO,
                  pre_smote_multiplier=PRE_SMOTE_MAJORITY_MULTIPLIER):
    """
    X: ndarray (n_instances, n_timesteps, n_features)
    y: ndarray (n_instances,) binary labels
    Returns balanced (X_bal, y_bal), same shape convention.
    """
    n_inst, n_time, n_feat = X.shape
    X_flat = X.reshape(n_inst, -1)

    pos = int(y.sum())
    neg = len(y) - pos
    print(f"Before balancing: {pos} positive / {neg} negative "
          f"(ratio={pos / max(neg, 1):.4f})")

    if pos == 0:
        raise ValueError("No positive samples in training set — cannot balance.")

    # ---- Step 1: pre-undersample majority BEFORE SMOTE (memory guard) ----
    target_majority = min(neg, pos * pre_smote_multiplier)
    if target_majority < neg:
        pre_rus = RandomUnderSampler(
            sampling_strategy={0: target_majority, 1: pos},
            random_state=RANDOM_SEED,
        )
        X_flat, y = pre_rus.fit_resample(X_flat, y)
        neg = target_majority
        print(f"Pre-undersampled majority to {neg} (x{pre_smote_multiplier} minority) "
              f"before SMOTE to control memory use.")

    # Guard: SMOTE requires n_neighbors < n_minority_samples
    k_neighbors = min(5, max(1, pos - 1))

    try:
        smote = SMOTE(sampling_strategy=smote_ratio, random_state=RANDOM_SEED,
                       k_neighbors=k_neighbors)
        X_over, y_over = smote.fit_resample(X_flat, y)
    except ValueError as e:
        print(f"SMOTE skipped ({e}); proceeding without oversampling.")
        X_over, y_over = X_flat, y

    del X_flat

    rus = RandomUnderSampler(sampling_strategy=rus_ratio, random_state=RANDOM_SEED)
    X_bal, y_bal = rus.fit_resample(X_over, y_over)

    del X_over, y_over

    X_bal = X_bal.reshape(-1, n_time, n_feat).astype(np.float32)

    pos_b = int(y_bal.sum())
    neg_b = len(y_bal) - pos_b
    print(f"After balancing:  {pos_b} positive / {neg_b} negative "
          f"(ratio={pos_b / max(neg_b, 1):.4f})")

    return X_bal, y_bal


def compute_class_weights(y):
    """Inverse-frequency class weights, for use with weighted CrossEntropyLoss."""
    classes, counts = np.unique(y, return_counts=True)
    total = len(y)
    weights = total / (len(classes) * counts)
    weight_dict = {int(c): float(w) for c, w in zip(classes, weights)}
    print(f"Class weights: {weight_dict}")
    return weight_dict

Overwriting sampling.py


In [6]:
%%writefile model.py
"""
model.py
--------
Temporal Convolutional Network (TCN) with an attention-pooling head for
SWAN-SF flare classification, extended to MULTI-TASK LEARNING:

  Task 1 (primary):   binary flare classification, as before.
  Task 2 (auxiliary):  self-supervised reconstruction of the input MVTS
                        sequence from the SAME pooled representation used
                        for classification.

Why this auxiliary task: the attention-pooled vector is a serious
bottleneck (e.g. 64 numbers reconstructing a 60-timestep x N-feature
sequence). Forcing that bottleneck to retain enough information to
rebuild the original series regularizes the shared TCN backbone toward
general, information-rich temporal representations, rather than
representations that only capture whatever narrow signal happens to
separate the (very rare) positive class in the current training fold.
This is a standard multi-task / self-supervised auxiliary-loss strategy
in representation learning, and — unlike adding a flare-intensity
auxiliary head — it needs no extra labels, so it is completely
unaffected by SMOTE/undersampling bookkeeping in sampling.py.
"""

import torch
import torch.nn as nn
from config import TCN_CHANNELS, KERNEL_SIZE, DROPOUT, N_CLASSES


class TemporalBlock(nn.Module):
    """One dilated causal-convolution residual block."""

    def __init__(self, in_ch, out_ch, kernel_size, dilation, dropout=DROPOUT):
        super().__init__()
        padding = (kernel_size - 1) * dilation
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size,
                                padding=padding, dilation=dilation)
        self.bn1 = nn.BatchNorm1d(out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size,
                                padding=padding, dilation=dilation)
        self.bn2 = nn.BatchNorm1d(out_ch)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.downsample = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else None
        self.padding = padding

    def forward(self, x):
        out = self.conv1(x)
        out = out[:, :, :x.size(2)]
        out = self.relu(self.bn1(out))
        out = self.dropout(out)

        out = self.conv2(out)
        out = out[:, :, :x.size(2)]
        out = self.relu(self.bn2(out))
        out = self.dropout(out)

        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)


class AttentionPool(nn.Module):
    """Learns a scalar attention weight per timestep, then weighted-sums over time."""

    def __init__(self, dim):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(dim, dim // 2),
            nn.Tanh(),
            nn.Linear(dim // 2, 1),
        )

    def forward(self, x):
        scores = self.attn(x)
        weights = torch.softmax(scores, dim=1)
        pooled = (x * weights).sum(dim=1)
        return pooled, weights.squeeze(-1)


class ReconstructionDecoder(nn.Module):
    """
    Auxiliary head: maps the pooled representation back to the full
    (n_timesteps, n_features) input sequence. A deliberately narrow
    bottleneck -> wide reconstruction target, so the model must retain
    genuinely informative temporal structure to succeed at this task.
    """

    def __init__(self, pooled_dim, n_timesteps, n_features, hidden_mult=4):
        super().__init__()
        hidden = pooled_dim * hidden_mult
        self.n_timesteps = n_timesteps
        self.n_features = n_features
        self.net = nn.Sequential(
            nn.Linear(pooled_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, n_timesteps * n_features),
        )

    def forward(self, pooled):
        out = self.net(pooled)
        return out.view(-1, self.n_timesteps, self.n_features)


class TCNAttentionModel(nn.Module):
    def __init__(self, n_features, n_timesteps, n_classes=N_CLASSES, channels=None,
                 kernel_size=KERNEL_SIZE, dropout=DROPOUT):
        super().__init__()
        channels = channels or TCN_CHANNELS

        layers = []
        in_ch = n_features
        for i, out_ch in enumerate(channels):
            layers.append(TemporalBlock(in_ch, out_ch, kernel_size,
                                         dilation=2 ** i, dropout=dropout))
            in_ch = out_ch
        self.tcn = nn.Sequential(*layers)

        self.attention = AttentionPool(channels[-1])
        self.classifier = nn.Sequential(
            nn.Linear(channels[-1], channels[-1] // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(channels[-1] // 2, n_classes),
        )
        # Multi-task auxiliary head (self-supervised reconstruction)
        self.decoder = ReconstructionDecoder(channels[-1], n_timesteps, n_features)

    def forward(self, x):
        # x: (batch, timesteps, features) -> conv expects (batch, features, timesteps)
        x_in = x
        x = x.permute(0, 2, 1)
        feats = self.tcn(x)
        feats = feats.permute(0, 2, 1)
        pooled, attn_weights = self.attention(feats)
        logits = self.classifier(pooled)
        reconstruction = self.decoder(pooled)
        return logits, attn_weights, reconstruction


def build_model(n_features, n_timesteps, device):
    model = TCNAttentionModel(n_features=n_features, n_timesteps=n_timesteps)
    return model.to(device)

Overwriting model.py


In [7]:
%%writefile evaluate.py
"""
evaluate.py
-----------
Solar-flare-specific evaluation metrics (TSS, HSS) plus standard
classification metrics (accuracy, precision, recall, F1, AUC), robust
to SWAN-SF's extreme class imbalance.

Key addition: since TSS = TPR - FPR (the Youden's J statistic), it is
threshold-dependent. Rather than always using the default 0.5 decision
threshold (argmax), `find_best_threshold` scans thresholds on the
VALIDATION set to find the one that maximizes TSS there, and that
threshold is then applied to the held-out TEST set. This is a purely
post-hoc decision-boundary adjustment — no leakage, since the threshold
is chosen using only validation data.
"""

import numpy as np
import torch
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score,
)


def true_skill_statistic(y_true, y_pred):
    """TSS = TPR - FPR. Standard skill metric in solar flare forecasting.
    Range: [-1, 1], 0 = no skill, 1 = perfect."""
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    return tpr - fpr


def heidke_skill_score(y_true, y_pred):
    """HSS: skill relative to random chance forecast. Range: (-inf, 1]."""
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    n = tp + tn + fp + fn
    if n == 0:
        return 0.0
    expected_correct = ((tp + fp) * (tp + fn) + (tn + fn) * (tn + fp)) / n
    denom = n - expected_correct
    if denom == 0:
        return 0.0
    return (tp + tn - expected_correct) / denom


@torch.no_grad()
def run_inference(model, data_loader, device):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    for X, y in data_loader:
        X = X.to(device)
        logits, _, _ = model(X)   # (logits, attention_weights, reconstruction)
        probs = torch.softmax(logits, dim=1)[:, 1]
        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.numpy())
        all_probs.extend(probs.cpu().numpy())

    return (np.array(all_labels), np.array(all_preds), np.array(all_probs))


def find_best_threshold(y_true, y_prob, thresholds=None):
    """
    Scan candidate thresholds and return the one that maximizes TSS.
    Intended to be called on the VALIDATION set only; the resulting
    threshold is then applied to the test set in evaluate().
    """
    if thresholds is None:
        thresholds = np.linspace(0.01, 0.99, 99)

    best_t, best_tss = 0.5, -1.0
    for t in thresholds:
        preds = (y_prob >= t).astype(int)
        tss = true_skill_statistic(y_true, preds)
        if tss > best_tss:
            best_tss = tss
            best_t = float(t)

    return best_t, best_tss


def evaluate(model, data_loader, device, partition_name="test", verbose=True, threshold=0.5):
    """
    threshold: decision threshold applied to P(positive). Default 0.5
    reproduces plain argmax behavior. Pass a validation-tuned threshold
    (from find_best_threshold) to evaluate with a TSS-optimized cutoff.
    """
    y_true, _, y_prob = run_inference(model, data_loader, device)
    y_pred = (y_prob >= threshold).astype(int)

    tss = true_skill_statistic(y_true, y_pred)
    hss = heidke_skill_score(y_true, y_pred)
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = float("nan")

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    results = {
        "partition": partition_name,
        "threshold": threshold,
        "TSS": tss,
        "HSS": hss,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc,
        "confusion_matrix": cm.tolist(),
    }

    if verbose:
        print(f"\n--- Evaluation on {partition_name} (threshold={threshold:.3f}) ---")
        print(f"TSS: {tss:.4f}  |  HSS: {hss:.4f}  |  AUC: {auc:.4f}")
        print(f"Accuracy: {accuracy:.4f}  |  Precision: {precision:.4f}  "
              f"|  Recall: {recall:.4f}  |  F1: {f1:.4f}")
        print(f"Confusion matrix [[TN, FP],[FN, TP]]:\n{cm}")

    return results


def summarize_cross_partition_results(all_results):
    """Aggregate all metrics across the rotating-partition CV folds."""
    metric_names = ["TSS", "HSS", "accuracy", "precision", "recall", "f1", "auc"]
    metric_vals = {m: [r[m] for r in all_results] for m in metric_names}

    print("\n===== Cross-Partition Summary =====")
    header = f"  {'Fold':>6s} | " + " | ".join(f"{m:>9s}" for m in metric_names)
    print(header)
    print("  " + "-" * (len(header) - 2))
    for r in all_results:
        row = f"  {r['partition']:>6s} | " + " | ".join(f"{r[m]:9.4f}" for m in metric_names)
        print(row)

    print("  " + "-" * (len(header) - 2))
    summary = {}
    for m in metric_names:
        vals = [v for v in metric_vals[m] if not np.isnan(v)]
        mean_v = float(np.mean(vals)) if vals else float("nan")
        std_v = float(np.std(vals)) if vals else float("nan")
        summary[f"mean_{m}"] = mean_v
        summary[f"std_{m}"] = std_v
        print(f"  Mean {m:>9s}: {mean_v:.4f} +/- {std_v:.4f}")

    return summary

Overwriting evaluate.py


In [8]:
%%writefile main.py
"""
main.py
-------
End-to-end SWAN-SF pipeline: preprocessing -> sampling -> TCN+Attention
model -> evaluation, using the standard rotating-partition protocol
(train on 4 partitions, test on the held-out one, repeat for all 5).

TSS-focused version:
  - During training, the "best" checkpoint is selected by VALIDATION TSS
    (not validation loss). Loss and TSS don't always move together under
    heavy class imbalance, and TSS is the metric that actually matters
    for this task.
  - After training, a decision threshold is tuned on the validation set
    to maximize TSS there (TSS = TPR - FPR = Youden's J, which is
    threshold-dependent), and that same threshold is then applied when
    evaluating on the held-out TEST partition. No leakage: threshold is
    chosen using validation data only.

Memory-safe: partitions are loaded from cache fresh per fold, and large
arrays are freed as soon as they're no longer needed.

Run:
    python main.py
"""

import os
import gc
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_auc_score

import config
from swan_sf_loader import load_raw_partition_cached
from preprocessing import (
    impute_missing, cap_outliers, select_features,
    apply_feature_selection, fit_scaler, apply_scaler,
)
from sampling import balance_mvts, compute_class_weights
from model import build_model
from evaluate import (
    evaluate, summarize_cross_partition_results,
    true_skill_statistic, run_inference, find_best_threshold,
)


class FocalLoss(nn.Module):
    """
    Focal Loss (Lin et al., 2017), adapted for extreme class imbalance.
    Down-weights easy, already-well-classified examples and focuses
    training on hard/misclassified ones — often gives a better
    precision/recall trade-off than plain class-weighted cross-entropy
    on severely imbalanced data like SWAN-SF.

    loss = -alpha_t * (1 - p_t)^gamma * log(p_t)
    """

    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha  # tensor of per-class weights, or None
        self.gamma = gamma

    def forward(self, logits, targets):
        log_probs = torch.log_softmax(logits, dim=1)
        probs = torch.exp(log_probs)
        log_pt = log_probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        pt = probs.gather(1, targets.unsqueeze(1)).squeeze(1)

        focal_term = (1 - pt) ** self.gamma
        loss = -focal_term * log_pt

        if self.alpha is not None:
            alpha_t = self.alpha[targets]
            loss = alpha_t * loss

        return loss.mean()


def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def make_loader(X, y, batch_size, shuffle):
    X_t = torch.tensor(X, dtype=torch.float32)
    y_t = torch.tensor(y, dtype=torch.long)
    ds = TensorDataset(X_t, y_t)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


def load_partition(name):
    cache_path = os.path.join(config.CACHE_DIR, f"{name}.pkl")
    X, y, ids, feature_names = load_raw_partition_cached(
        config.PARTITION_DIRS[name], cache_path, n_timesteps=config.N_TIMESTEPS
    )
    return X, y, feature_names


def preprocess_one(X, keep_idx, scaler):
    X = impute_missing(X)
    X = cap_outliers(X)
    X_sel = apply_feature_selection(X, keep_idx)
    del X
    gc.collect()
    X_final = apply_scaler(X_sel, scaler)
    del X_sel
    gc.collect()
    return X_final


def train_one_fold(X_train, y_train, X_val, y_val, n_features, device, class_weight_dict=None):
    n_timesteps = X_train.shape[1]
    model = build_model(n_features, n_timesteps, device)

    if config.USE_CLASS_WEIGHTED_LOSS and class_weight_dict is not None:
        weight_tensor = torch.tensor(
            [class_weight_dict.get(0, 1.0), class_weight_dict.get(1, 1.0)],
            dtype=torch.float32,
        ).to(device)
    else:
        weight_tensor = None

    if getattr(config, "USE_FOCAL_LOSS", False):
        ce_criterion = FocalLoss(alpha=weight_tensor, gamma=config.FOCAL_GAMMA)
    elif weight_tensor is not None:
        ce_criterion = nn.CrossEntropyLoss(weight=weight_tensor)
    else:
        ce_criterion = nn.CrossEntropyLoss()
    recon_criterion = nn.MSELoss()

    optimizer = torch.optim.Adam(model.parameters(), lr=config.LEARNING_RATE,
                                  weight_decay=config.WEIGHT_DECAY)

    scheduler_type = getattr(config, "LR_SCHEDULER", "plateau")
    if scheduler_type == "cosine":
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=config.EPOCHS
        )
    else:
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.5, patience=3
        )

    train_loader = make_loader(X_train, y_train, config.BATCH_SIZE, shuffle=True)
    val_loader = make_loader(X_val, y_val, config.BATCH_SIZE, shuffle=False)

    best_val_tss = -1.0
    best_state = None
    patience_counter = 0

    for epoch in range(1, config.EPOCHS + 1):
        model.train()
        train_loss, train_ce, train_recon = 0.0, 0.0, 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            logits, _, reconstruction = model(X_batch)
            ce_loss = ce_criterion(logits, y_batch)
            recon_loss = recon_criterion(reconstruction, X_batch)
            loss = ce_loss + config.RECON_LOSS_WEIGHT * recon_loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            train_loss += loss.item() * X_batch.size(0)
            train_ce += ce_loss.item() * X_batch.size(0)
            train_recon += recon_loss.item() * X_batch.size(0)
        train_loss /= len(train_loader.dataset)
        train_ce /= len(train_loader.dataset)
        train_recon /= len(train_loader.dataset)

        model.eval()
        val_loss = 0.0
        all_val_probs, all_val_labels = [], []
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                logits, _, reconstruction = model(X_batch)
                ce_loss = ce_criterion(logits, y_batch)
                recon_loss = recon_criterion(reconstruction, X_batch)
                loss = ce_loss + config.RECON_LOSS_WEIGHT * recon_loss
                val_loss += loss.item() * X_batch.size(0)
                probs = torch.softmax(logits, dim=1)[:, 1]
                all_val_probs.extend(probs.cpu().numpy())
                all_val_labels.extend(y_batch.cpu().numpy())
        val_loss /= len(val_loader.dataset)
        val_labels_arr = np.array(all_val_labels)
        val_probs_arr = np.array(all_val_probs)
        # NOTE: train set is SMOTE/RUS-balanced but val set keeps the natural
        # imbalance, so a fixed 0.5 argmax threshold under-represents the
        # model's real separating power and drifts as training progresses.
        # Tune the threshold on validation probabilities each epoch instead,
        # so checkpoint selection reflects the model's underlying skill.
        _, val_tss = find_best_threshold(val_labels_arr, val_probs_arr)
        # AUC is threshold-independent — track it alongside TSS. If AUC keeps
        # improving while val_TSS falls, that points to a calibration/
        # threshold issue rather than the model actually getting worse.
        try:
            val_auc = roc_auc_score(val_labels_arr, val_probs_arr)
        except ValueError:
            val_auc = float("nan")  # only one class present in this val batch set

        if scheduler_type == "cosine":
            scheduler.step()
        else:
            scheduler.step(val_loss)

        print(f"  Epoch {epoch:3d}/{config.EPOCHS}  "
              f"train_loss={train_loss:.4f} (ce={train_ce:.4f} recon={train_recon:.4f})  "
              f"val_loss={val_loss:.4f}  val_TSS={val_tss:.4f}  val_AUC={val_auc:.4f}")

        if val_tss > best_val_tss:
            best_val_tss = val_tss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= config.EARLY_STOP_PATIENCE:
                print(f"  Early stopping at epoch {epoch} (no val TSS improvement "
                      f"for {config.EARLY_STOP_PATIENCE} epochs). Best val_TSS={best_val_tss:.4f}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    del train_loader, optimizer, scheduler, ce_criterion, recon_criterion
    gc.collect()

    return model, val_loader


def run_rotating_partition_cv():
    set_seed(config.RANDOM_SEED)
    os.makedirs(config.OUTPUT_DIR, exist_ok=True)
    device = torch.device(config.DEVICE if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    partition_names = list(config.PARTITION_DIRS.keys())

    print("\n=== Pre-caching all partitions (raw SWAN-SF format) ===")
    for name in partition_names:
        X, y, _ = load_partition(name)
        del X, y
    gc.collect()

    all_results = []

    for test_name in partition_names:
        ckpt_path = os.path.join(config.OUTPUT_DIR, f"model_fold_{test_name}.pt")
        if os.path.exists(ckpt_path):
            print(f"\nFold '{test_name}' already has a saved checkpoint "
                  f"({ckpt_path}) — skipping (delete the file to re-run this fold).")
            continue

        print(f"\n============================================")
        print(f"  Fold: hold out '{test_name}' as TEST partition")
        print(f"============================================")

        train_names = [p for p in partition_names if p != test_name]
        val_name = train_names[0]
        fit_train_names = train_names[1:]

        print("\n--- Loading + preprocessing TRAIN partitions (from cache) ---")
        train_Xs, train_ys = [], []
        feature_names = None
        for name in fit_train_names:
            X, y, feature_names = load_partition(name)
            train_Xs.append(X)
            train_ys.append(y)

        X_train_raw = np.concatenate(train_Xs, axis=0)
        y_train_raw = np.concatenate(train_ys, axis=0)
        del train_Xs, train_ys
        gc.collect()

        X_train_raw = impute_missing(X_train_raw)
        X_train_raw = cap_outliers(X_train_raw)

        keep_idx, kept_feats = select_features(X_train_raw, feature_names)
        X_train_sel = apply_feature_selection(X_train_raw, keep_idx)
        del X_train_raw
        gc.collect()

        scaler = fit_scaler(X_train_sel)
        X_train_p = apply_scaler(X_train_sel, scaler)
        del X_train_sel
        gc.collect()

        print("\n--- Loading + preprocessing VAL partition ---")
        X_val_raw, y_val_raw, _ = load_partition(val_name)
        X_val_p = preprocess_one(X_val_raw, keep_idx, scaler)
        del X_val_raw
        gc.collect()

        print("--- Loading + preprocessing TEST partition ---")
        X_test_raw, y_test_raw, _ = load_partition(test_name)
        X_test_p = preprocess_one(X_test_raw, keep_idx, scaler)
        del X_test_raw
        gc.collect()

        if getattr(config, "USE_SMOTE_BALANCING", True):
            print("\n--- Balancing training set (SMOTE + undersampling) ---")
            X_train_bal, y_train_bal = balance_mvts(X_train_p, y_train_raw)
        else:
            # Train on the REAL class distribution — no synthetic samples.
            # Class imbalance is instead handled purely through
            # class-weighted Focal Loss (see USE_CLASS_WEIGHTED_LOSS /
            # USE_FOCAL_LOSS in config.py). This keeps the train distribution
            # aligned with val/test, avoiding the synthetic-pattern mismatch
            # SMOTE can introduce on multivariate time series.
            print("\n--- Skipping SMOTE/undersampling (USE_SMOTE_BALANCING=False) "
                  "— training on real class distribution ---")
            X_train_bal, y_train_bal = X_train_p, y_train_raw
        del X_train_p, y_train_raw
        gc.collect()
        class_weights = compute_class_weights(y_train_bal)

        print("\n--- Training (model selection by validation TSS) ---")
        n_features = X_train_bal.shape[2]
        model, val_loader = train_one_fold(
            X_train_bal, y_train_bal, X_val_p, y_val_raw,
            n_features, device, class_weight_dict=class_weights,
        )
        del X_train_bal, y_train_bal
        gc.collect()

        # ---- Tune decision threshold on VALIDATION set (maximize TSS) ----
        val_true, _, val_prob = run_inference(model, val_loader, device)
        best_threshold, best_val_tss = find_best_threshold(val_true, val_prob)
        print(f"\nValidation-tuned threshold: {best_threshold:.3f} "
              f"(val TSS at this threshold = {best_val_tss:.4f})")
        del val_loader, X_val_p, y_val_raw
        gc.collect()

        # ---- Evaluation on held-out test partition, using tuned threshold ----
        test_loader = make_loader(X_test_p, y_test_raw, config.BATCH_SIZE, shuffle=False)
        result = evaluate(model, test_loader, device, partition_name=test_name,
                           threshold=best_threshold)
        all_results.append(result)

        torch.save(model.state_dict(), ckpt_path)
        print(f"Saved model checkpoint -> {ckpt_path}")

        del X_test_p, y_test_raw, test_loader, model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    summary = summarize_cross_partition_results(all_results)

    results_path = os.path.join(config.OUTPUT_DIR, "cv_results.json")
    with open(results_path, "w") as f:
        json.dump({"folds": all_results, "summary": summary}, f, indent=2)
    print(f"\nSaved full results -> {results_path}")

    return all_results, summary


# NOTE: the `if __name__ == "__main__":` auto-run guard was removed on
# purpose. If this file is pasted straight into a notebook cell, that
# guard evaluates to True (a notebook cell's __name__ is "__main__") and
# the whole pipeline fires immediately when you run the cell — which is
# what was happening. Call it explicitly, in its own cell, instead:
#
#   from main import run_rotating_partition_cv
#   all_results, summary = run_rotating_partition_cv()
#
# (If you later use %%writefile main.py + import main, this file's
# __name__ becomes "main", not "__main__", so the guard would have been
# safe in that flow too — but leaving it out avoids the footgun either way.)

Overwriting main.py


## 3. Get your SWAN-SF data onto the VM

Download SWAN-SF from Harvard Dataverse: https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/EBCFKM

It should end up as a folder containing `partition1` .. `partition5`, where each partition holds thousands of tab-delimited instance files with `tag[value]` filename metadata (e.g. `lab[M]_id[1201]_st[...]_et[...].csv`). Pick **ONE** option below to get it into `/content/SWAN-SF` (the default `config.py` path on Colab).

In [9]:
# --- OPTION B: SWAN-SF already sitting in Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

import tarfile, os
import config

drive_dir = "/content/drive/MyDrive/SWAN-SF"
os.makedirs("/content/SWAN-SF", exist_ok=True)

for i in range(1, 6):
    target = f"/content/SWAN-SF/partition{i}"
    if os.path.isdir(target) and len(os.listdir(target)) > 0:
        print(f"partition{i} already extracted — skipping.")
        continue
    file_path = f"{drive_dir}/partition{i}_instances.tar.gz"
    print(f"Extracting partition{i} ...")
    with tarfile.open(file_path) as tf:
        tf.extractall("/content/SWAN-SF")

print("Done!")
!ls /content/SWAN-SF

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
partition1 already extracted — skipping.
Extracting partition2 ...


/tmp/ipykernel_4247/37162951.py:19: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall("/content/SWAN-SF")


Extracting partition3 ...
Extracting partition4 ...
Extracting partition5 ...
Done!
partition1  partition2	partition3  partition4	partition5


 4. Check GPU + confirm config paths

In [10]:
import os, torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected — Runtime > Change runtime type > select a GPU for faster training.")

import importlib, config
importlib.reload(config)
print("\nRAW_DATA_ROOT:", config.RAW_DATA_ROOT)
for name, path in config.PARTITION_DIRS.items():
    exists = os.path.isdir(path)
    print(f"  {name}: {path}  {'OK' if exists else 'MISSING'}")

CUDA available: True
GPU: Tesla T4

RAW_DATA_ROOT: /content/SWAN-SF
  p1: /content/SWAN-SF/partition1  OK
  p2: /content/SWAN-SF/partition2  OK
  p3: /content/SWAN-SF/partition3  OK
  p4: /content/SWAN-SF/partition4  OK
  p5: /content/SWAN-SF/partition5  OK


## 5. (Optional) Quick smoke-test settings

Uncomment to shrink epochs for a fast test run before committing to full training.

In [11]:
#config.EPOCHS = 3
#config.BATCH_SIZE = 128

## 6. Run the full pipeline

Loads all 5 partitions (cached after first parse), then runs rotating-partition cross-validation: preprocess → balance → train TCN+Attention → evaluate, holding out each partition as test in turn.

In [12]:
from main import run_rotating_partition_cv
all_results, summary = run_rotating_partition_cv()

Using device: cuda

=== Pre-caching all partitions (raw SWAN-SF format) ===


Loading partition1:  27%|██▋       | 5018/18785 [01:23<03:48, 60.19it/s]


KeyboardInterrupt: 

## 7. Inspect results

In [ ]:
print("Mean TSS: {:.4f} +/- {:.4f}".format(summary["mean_TSS"], summary["std_TSS"]))
print("Mean HSS: {:.4f} +/- {:.4f}".format(summary["mean_HSS"], summary["std_HSS"]))
print("Mean Accuracy: {:.4f} +/- {:.4f}".format(summary["mean_accuracy"], summary["std_accuracy"]))
print("Mean Precision: {:.4f} +/- {:.4f}".format(summary["mean_precision"], summary["std_precision"]))
print("Mean Recall: {:.4f} +/- {:.4f}".format(summary["mean_recall"], summary["std_recall"]))
print("Mean F1: {:.4f} +/- {:.4f}".format(summary["mean_f1"], summary["std_f1"]))
print("Mean AUC: {:.4f} +/- {:.4f}".format(summary["mean_auc"], summary["std_auc"]))

for r in all_results:
    print(f"  {r['partition']}: TSS={r['TSS']:.4f}  HSS={r['HSS']:.4f}  "
          f"Acc={r['accuracy']:.4f}  Prec={r['precision']:.4f}  "
          f"Rec={r['recall']:.4f}  F1={r['f1']:.4f}")

In [ ]:
import glob, os
for f in glob.glob("/content/swan_sf_outputs/model_fold_*.pt"):
    os.remove(f)
    print("Removed:", f)
print("Ready for fresh full run.")

## 8. Download trained model checkpoints + results

In [ ]:
import shutil
shutil.make_archive('/content/swan_sf_results', 'zip', config.OUTPUT_DIR)

from google.colab import files
files.download('/content/swan_sf_results.zip')